# 012S AI Image Generation System v1

Colab launcher for ComfyUI. Select a GPU runtime (A100 recommended), then run the cells from top to bottom. Model switches are read from `config/system_config.json`; Flux is enabled and SDXL is optional.

Product Key Visual and batch workflows require a transparent product PNG. The runner centers smaller source PNGs on the selected canvas without resizing their product pixels.


In [ ]:
# STEP 1 — Mount Google Drive and make the project scripts available.
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive', force_remount=False)
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Google Drive did not mount at /content/drive/MyDrive. Rerun this cell and approve Drive access.')
BOOTSTRAP_DRIVE_ROOT = Path('/content/drive/MyDrive/012s-image-system')
REPOSITORY_URL_FILE = BOOTSTRAP_DRIVE_ROOT / 'metadata' / 'repository_url.txt'
DEFAULT_REPOSITORY_URL = 'https://github.com/alberthuang-012S/colab-A100.git'
REPOSITORY_URL_FILE.parent.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path(os.environ.get('012S_PROJECT_DIR', '/content/012s-image-system'))
if not (PROJECT_DIR / 'scripts' / 'environment_check.py').is_file():
    repository_url = os.environ.get('012S_REPOSITORY_URL', '').strip()
    if not repository_url and REPOSITORY_URL_FILE.is_file():
        repository_url = REPOSITORY_URL_FILE.read_text(encoding='utf-8').strip()
    if not repository_url:
        repository_url = DEFAULT_REPOSITORY_URL
    from urllib.parse import urlsplit
    parsed_repository_url = urlsplit(repository_url)
    if parsed_repository_url.scheme in ('http', 'https') and (parsed_repository_url.username or parsed_repository_url.password):
        raise ValueError('Do not put credentials in the Git URL. Use your configured Git authentication instead.')
    REPOSITORY_URL_FILE.write_text(repository_url + '\n', encoding='utf-8')
    subprocess.run(['git', 'clone', '--depth', '1', repository_url, str(PROJECT_DIR)], check=True)
sys.path.insert(0, str(PROJECT_DIR))
from scripts.common import load_config, setup_logging
from scripts.drive_manager import ensure_drive_layout, get_drive_root
CONFIG_PATH = PROJECT_DIR / 'config' / 'system_config.json'
CONFIG = load_config(CONFIG_PATH)
DRIVE_ROOT = get_drive_root(CONFIG)
ensure_drive_layout(DRIVE_ROOT)
os.environ['012S_DRIVE_ROOT'] = str(DRIVE_ROOT)
LOGGER = setup_logging(DRIVE_ROOT / 'logs')
print(f'Drive project folder ready: {DRIVE_ROOT}')
print(f'Launcher source: {PROJECT_DIR}')

In [ ]:
# STEP 2 — Environment check. Stop before model installation if GPU/CUDA is unavailable.
from scripts.environment_check import detect_environment, format_report
ENV_REPORT = detect_environment(CONFIG, DRIVE_ROOT)
print(format_report(ENV_REPORT))
if ENV_REPORT.status != 'READY':
    raise RuntimeError('Environment is not READY. Select a GPU runtime, enable CUDA, and mount Drive before continuing.')

In [ ]:
# STEP 3 — Clone ComfyUI on a clean runtime, or fast-forward the existing checkout.
from scripts.comfy_manager import ensure_comfyui_repo
COMFY_DIR = Path(CONFIG.get('comfyui_dir', '/content/ComfyUI'))
COMFY_DIR = ensure_comfyui_repo(CONFIG['comfyui_repo_url'], COMFY_DIR, LOGGER)
print(f'ComfyUI source ready: {COMFY_DIR}')

In [ ]:
# STEP 4 — Install ComfyUI requirements for this runtime.
from scripts.comfy_manager import install_requirements
install_requirements(COMFY_DIR, logger=LOGGER)
ENV_REPORT = detect_environment(CONFIG, DRIVE_ROOT)
print(format_report(ENV_REPORT))
if ENV_REPORT.status != 'READY':
    raise RuntimeError('CUDA/Drive verification failed after dependency setup. Review the Environment Report before downloading models.')
print('ComfyUI dependencies are ready.')

In [ ]:
# STEP 5 — Check the selected model files already stored on Drive.
from scripts.model_manager import selected_models, model_cached_valid
INSTALL_FLUX = bool(CONFIG.get('install_flux', True))
INSTALL_SDXL = bool(CONFIG.get('install_sdxl', False))
SELECTED_MODELS = selected_models(CONFIG, install_flux=INSTALL_FLUX, install_sdxl=INSTALL_SDXL)
for category, models in SELECTED_MODELS.items():
    for model in models:
        path = DRIVE_ROOT / 'models' / category / model['filename']
        state = 'READY' if model_cached_valid(path, model, CONFIG.get('model_sha256', {}).get(model['filename'])) else 'MISSING'
        print(f'{state}: {path}')

In [ ]:
# STEP 6 — Download only selected models that are missing or fail validation.
from scripts.model_manager import sync_models
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass  # Public model downloads may work without a Hugging Face token.
MODEL_REPORT = sync_models(CONFIG, DRIVE_ROOT, install_flux=INSTALL_FLUX, install_sdxl=INSTALL_SDXL, logger=LOGGER)
print('Installed:', MODEL_REPORT.installed or 'None')
print('Skipped:', MODEL_REPORT.skipped or 'None')
for message in MODEL_REPORT.auth_required:
    print('AUTH REQUIRED:', message)
for message in MODEL_REPORT.failed:
    print('ERROR SUMMARY:', message)
if MODEL_REPORT.auth_required:
    print('Add HF_TOKEN in Colab Secrets, accept any required model terms, then rerun this cell.')

In [ ]:
# STEP 7 — Configure ComfyUI to read model files directly from persistent Drive storage.
from scripts.drive_manager import install_extra_model_paths
MODEL_PATH_CONFIG = install_extra_model_paths(COMFY_DIR, DRIVE_ROOT)
print(f'ComfyUI model paths configured: {MODEL_PATH_CONFIG}')

In [ ]:
# STEP 8 — Prepare the custom-node folder. Phase 1 workflows use ComfyUI core nodes only.
CUSTOM_NODES_DIR = COMFY_DIR / 'custom_nodes'
CUSTOM_NODES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Custom nodes directory ready: {CUSTOM_NODES_DIR}')
print('No third-party custom nodes are required for the included workflows.')

In [ ]:
# STEP 9 — Copy workflow JSON and prompt templates to their persistent Drive folders.
from scripts.workflow_manager import install_workflows, load_registry
SYNCED_FILES = install_workflows(DRIVE_ROOT)
print(f'{len(SYNCED_FILES)} workflow/prompt files copied; existing Drive customizations were kept.')
for workflow_id, entry in load_registry()['workflows'].items():
    print(f"{workflow_id}: {entry['title']}")

In [ ]:
# STEP 10 — Start ComfyUI and wait for its API to respond.
import time, urllib.request
from scripts.comfy_manager import launch_comfyui
COMFY_LOG = DRIVE_ROOT / 'logs' / 'comfyui.log'
if 'COMFY_PROCESS' not in globals() or COMFY_PROCESS is None or COMFY_PROCESS.poll() is not None:
    COMFY_PROCESS = launch_comfyui(COMFY_DIR, CONFIG, background=True, log_path=COMFY_LOG)
api_url = f"http://127.0.0.1:{int(CONFIG.get('comfyui_port', 8188))}/system_stats"
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    try:
        with urllib.request.urlopen(api_url, timeout=5) as response:
            if response.status == 200:
                break
    except Exception:
        if COMFY_PROCESS and COMFY_PROCESS.poll() is not None:
            raise RuntimeError(f'ComfyUI exited. Review {COMFY_LOG}')
        time.sleep(3)
else:
    tail = COMFY_LOG.read_text(encoding='utf-8', errors='replace')[-4000:] if COMFY_LOG.exists() else 'No log yet.'
    raise TimeoutError(f'ComfyUI did not become ready. Recent log:\n{tail}')
print(f'ComfyUI API ready at http://127.0.0.1:{CONFIG.get("comfyui_port", 8188)}')
print(f'Log: {COMFY_LOG}')

In [ ]:
# STEP 11 — Show the ComfyUI web access URL.
from IPython.display import HTML, display
from google.colab import output
import html
UI_URL = output.eval_js(f"google.colab.kernel.proxyPort({int(CONFIG.get('comfyui_port', 8188))})")
display(HTML(f'<a href="{html.escape(UI_URL)}" target="_blank">Open 012S ComfyUI</a>'))
print(UI_URL)

## Run a named 012S workflow

The workflow runner submits the checked-in API graph to ComfyUI and stores each PNG plus its JSON metadata sidecar in Drive. The product and batch workflows take a transparent product PNG. For inpaint, pass both the source image and a black/white mask (white is regenerated).


In [ ]:
import subprocess

def run_012s_workflow(workflow, prompt, *, input_image=None, mask_image=None, seed=None,
                       width=1024, height=1024, count=1, product_name=None,
                       negative_prompt=None, category='draft'):
    command = [sys.executable, str(PROJECT_DIR / 'scripts' / 'run_workflow.py'),
               '--workflow', workflow, '--prompt', prompt, '--width', str(width),
               '--height', str(height), '--count', str(count), '--category', category,
               '--drive-root', str(DRIVE_ROOT), '--comfyui-dir', str(COMFY_DIR)]
    for flag, value in (('--input-image', input_image), ('--mask-image', mask_image),
                        ('--seed', seed), ('--product-name', product_name),
                        ('--negative-prompt', negative_prompt)):
        if value is not None:
            command.extend([flag, str(value)])
    return subprocess.run(command, cwd=PROJECT_DIR, check=False)

# Example: run_012s_workflow('text-to-image', 'Minimal premium apothecary bottle on pale stone', seed=12345)
# Product example: run_012s_workflow('product-kv', 'Bright limestone studio, soft side light', input_image='/content/drive/MyDrive/012s-image-system/assets/products/product.png', product_name='NNE', seed=12345)
# Batch example: run_012s_workflow('batch', 'Bright clean studio with limestone plinth', input_image='/content/drive/MyDrive/012s-image-system/assets/products/product.png', product_name='NNE', count=8, seed=1000)